In [18]:
import time

import torch
import torch.nn.utils.prune as prune
import torchvision.models as models
from torch.onnx import export

In [19]:
def model_to_onnx(model, output_file, input_shape = (1, 3, 224, 224)):
  model.eval()
  input_tensor = torch.randn(input_shape)
  input_names = ['input']
  output_names = ['output']
  torch.onnx.export(model, input_tensor, output_file, verbose = False, input_names = input_names, output_names = output_names)
  return output_file


In [20]:
#モデル読み込み
model = models.resnet50(weights = "IMAGENET1K_V2")


In [21]:
model_to_onnx(model, "resnet50_dense.onnx")   #onnx形式に

'resnet50_dense.onnx'

In [22]:
input_image = torch.ones((1, 3, 224, 224))
output = model(input_image)   #ウォームアップ

start_time = time.time()

with torch.no_grad():
  output = model(input_image)

end_time = time.time()
print(f"推論時間（密モデル）: {end_time - start_time:.4f} 秒")

推論時間（密モデル）: 0.0519 秒


In [23]:
# すべての畳み込み層を枝刈り対象に
parameters_to_prune = [
    (module, "weight") for module in model.modules() if isinstance(module, torch.nn.Conv2d)
]

In [24]:
# 大域的・非構造・強度枝刈り
prune.global_unstructured(
    parameters_to_prune,
    pruning_method = prune.L1Unstructured,
    amount = 0.9,
)

In [25]:
output = model   #ウォームアップ

start_time = time.time()

with torch.no_grad():
  output = model(input_image)

end_time = time.time()
print(f"推論時間（枝刈り直後）: {end_time - start_time:.4f} 秒")

推論時間（枝刈り直後）: 0.0747 秒


In [26]:
# 再訓練


In [27]:
# 永続化
for module in model.modules():
  if isinstance(module, torch.nn.Conv2d):
    prune.remove(module, "weight")

In [28]:
# ゼロ比率を表示
for module in model.modules():
  if isinstance(module, torch.nn.Conv2d):
    print(f"{module} Zero-Ratio: {100.0 * float(torch.sum(module.weight == 0)) / float(module.weight.nelement()):.2f}%")

Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False) Zero-Ratio: 40.20%
Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False) Zero-Ratio: 47.51%
Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False) Zero-Ratio: 68.17%
Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False) Zero-Ratio: 54.78%
Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False) Zero-Ratio: 51.41%
Conv2d(256, 64, kernel_size=(1, 1), stride=(1, 1), bias=False) Zero-Ratio: 56.46%
Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False) Zero-Ratio: 63.88%
Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False) Zero-Ratio: 55.66%
Conv2d(256, 64, kernel_size=(1, 1), stride=(1, 1), bias=False) Zero-Ratio: 53.83%
Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False) Zero-Ratio: 60.34%
Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False) Zero-Ratio: 60.27%
Conv2d(256, 128, kernel_size=(1, 1), str

In [29]:
output = model(input_image)   #ウォームアップ

start_time = time.time()

with torch.no_grad():
  output = model(input_image)

end_time = time.time()
print(f"推論時間（永続化後）: {end_time - start_time:.4f} 秒")

推論時間（永続化後）: 0.0518 秒


In [30]:
model_to_onnx(model, "resnet50_sparse.onnx")

'resnet50_sparse.onnx'

In [31]:
# 本当はdeepsparse使いたかったけど、Colabではつかえなかった。onnxruntimeはおそらく疎計算に対応していないから、denseとsparseで速度に違いが見られない。
import onnxruntime as ort
import numpy as np

# モデルパス
model_paths = ["resnet50_dense.onnx", "resnet50_sparse.onnx"]

# 入力サイズ（例: 1枚のカラー画像、224x224）
input_shape = (1, 3, 224, 224)
input_data = np.random.randn(*input_shape).astype(np.float32)

# 推論回数（平均を取るため）
num_runs = 50

# 比較ループ
for model_path in model_paths:
    # モデル読み込み
    session = ort.InferenceSession(model_path)
    input_name = session.get_inputs()[0].name

    # ウォームアップ（初回は遅くなるため）
    session.run(None, {input_name: input_data})

    # 時間計測
    start = time.time()
    for _ in range(num_runs):
        outputs = session.run(None, {input_name: input_data})
    end = time.time()

    avg_time = (end - start) / num_runs  # 秒単位
    print(f"{model_path}: 平均推論時間 = {avg_time:.4f} 秒（{num_runs}回平均）")


resnet50_dense.onnx: 平均推論時間 = 0.0227 秒（50回平均）
resnet50_sparse.onnx: 平均推論時間 = 0.0165 秒（50回平均）


In [32]:
def benchmark(model_path, input_shape=(1, 3, 224, 224), num_images=100):
    session = ort.InferenceSession(model_path)
    input_name = session.get_inputs()[0].name

    # ランダムな入力データを生成
    inputs = [np.random.randn(*input_shape).astype(np.float32) for _ in range(num_images)]

    # ウォームアップ（最初の遅延回避）
    session.run(None, {input_name: inputs[0]})

    # 推論時間を測定
    total_time = 0.0
    for inp in inputs:
        start = time.time()
        session.run(None, {input_name: inp})
        end = time.time()
        total_time += (end - start)

    avg_time = total_time / num_images
    print(f"{model_path}: 平均推論時間 {avg_time:.6f} 秒（{num_images}枚）")

# 使用例
benchmark("resnet50_dense.onnx")
benchmark("resnet50_sparse.onnx")


resnet50_dense.onnx: 平均推論時間 0.018702 秒（100枚）
resnet50_sparse.onnx: 平均推論時間 0.017929 秒（100枚）


In [33]:
# SSHした先の計算機がLinuxならつかえるはず。deepsparseはサービス終了しててLinuxでしかつかえないらしい。
#deepsparse=1.80と明示的にバージョン教えたらLinux上では動いた

In [34]:
from deepsparse.benchmark.benchmark_model import benchmark_model

print(benchmark_model("resnet50_dense.onnx", batch_size = 32))
print(benchmark_model("resnet50_sparse.onnx", batch_size = 32))

2025-06-14 16:09:27 deepsparse.benchmark.helpers INFO     Thread pinning to cores enabled
2025-06-14 16:09:31 deepsparse.benchmark.benchmark_model INFO     deepsparse.engine.Engine:
	onnx_file_path: resnet50_dense.onnx
	batch_size: 32
	num_cores: 20
	num_streams: 1
	scheduler: Scheduler.default
	fraction_of_supported_ops: 1.0
	cpu_avx_type: avx512
	cpu_vnni: True
2025-06-14 16:09:31 deepsparse.utils.onnx INFO     Generating input 'input', type = float32, shape = [32, 3, 224, 224]
2025-06-14 16:09:31 deepsparse.benchmark.benchmark_model INFO     Starting 'singlestream' performance measurements for 10 seconds
2025-06-14 16:09:43 deepsparse.benchmark.helpers INFO     Thread pinning to cores enabled


{'engine': 'deepsparse.engine.Engine:\n\tonnx_file_path: resnet50_dense.onnx\n\tbatch_size: 32\n\tnum_cores: 20\n\tnum_streams: 1\n\tscheduler: Scheduler.default\n\tfraction_of_supported_ops: 1.0\n\tcpu_avx_type: avx512\n\tcpu_vnni: True', 'version': '1.8.0', 'orig_model_path': 'resnet50_dense.onnx', 'model_path': 'resnet50_dense.onnx', 'batch_size': 32, 'input_shapes': None, 'num_cores': 20, 'scenario': 'singlestream', 'scheduler': 'Scheduler.default', 'seconds_to_run': 10, 'num_streams': 1, 'benchmark_result': {'scenario': 'singlestream', 'items_per_sec': 159.3821796372081, 'seconds_ran': 10.038763452990679, 'iterations': 50, 'median': 200.30386200232897, 'mean': 200.75069744023494, 'std': 1.4130153115084703, '25.0%': 199.96014301432297, '50.0%': 200.30386200232897, '75.0%': 201.06187000783393, '90.0%': 202.13246330386028, '95.0%': 203.52726559940493, '99.0%': 206.0037070090766, '99.9%': 206.0693943950755}, 'fraction_of_supported_ops': 1.0}


2025-06-14 16:09:47 deepsparse.benchmark.benchmark_model INFO     deepsparse.engine.Engine:
	onnx_file_path: resnet50_sparse.onnx
	batch_size: 32
	num_cores: 20
	num_streams: 1
	scheduler: Scheduler.default
	fraction_of_supported_ops: 1.0
	cpu_avx_type: avx512
	cpu_vnni: True
2025-06-14 16:09:47 deepsparse.utils.onnx INFO     Generating input 'input', type = float32, shape = [32, 3, 224, 224]
2025-06-14 16:09:47 deepsparse.benchmark.benchmark_model INFO     Starting 'singlestream' performance measurements for 10 seconds


{'engine': 'deepsparse.engine.Engine:\n\tonnx_file_path: resnet50_sparse.onnx\n\tbatch_size: 32\n\tnum_cores: 20\n\tnum_streams: 1\n\tscheduler: Scheduler.default\n\tfraction_of_supported_ops: 1.0\n\tcpu_avx_type: avx512\n\tcpu_vnni: True', 'version': '1.8.0', 'orig_model_path': 'resnet50_sparse.onnx', 'model_path': 'resnet50_sparse.onnx', 'batch_size': 32, 'input_shapes': None, 'num_cores': 20, 'scenario': 'singlestream', 'scheduler': 'Scheduler.default', 'seconds_to_run': 10, 'num_streams': 1, 'benchmark_result': {'scenario': 'singlestream', 'items_per_sec': 395.1152539237369, 'seconds_ran': 10.042639357998269, 'iterations': 124, 'median': 80.81471399054863, 'mean': 80.9617556691181, 'std': 0.887830355316614, '25.0%': 80.61398776771966, '50.0%': 80.81471399054863, '75.0%': 80.97743224789156, '90.0%': 81.7149307084037, '95.0%': 82.240574194293, '99.0%': 83.65809815149987, '99.9%': 87.81184994129585}, 'fraction_of_supported_ops': 1.0}


In [ ]:
results = benchmark_model("resnet50_dense.onnx", batch_size=32)
print(f"Total seconds ran: {results['benchmark_result']['seconds_ran']:.2f} s")
print(f"Iterations: {results['benchmark_result']['iterations']}")


2025-06-14 16:18:00 deepsparse.benchmark.helpers INFO     Thread pinning to cores enabled
2025-06-14 16:18:04 deepsparse.benchmark.benchmark_model INFO     deepsparse.engine.Engine:
	onnx_file_path: resnet50_dense.onnx
	batch_size: 32
	num_cores: 20
	num_streams: 1
	scheduler: Scheduler.default
	fraction_of_supported_ops: 1.0
	cpu_avx_type: avx512
	cpu_vnni: True
2025-06-14 16:18:04 deepsparse.utils.onnx INFO     Generating input 'input', type = float32, shape = [32, 3, 224, 224]
2025-06-14 16:18:04 deepsparse.benchmark.benchmark_model INFO     Starting 'singlestream' performance measurements for 10 seconds


Total seconds ran: 10.05 s
Iterations: 50


In [36]:
results = benchmark_model("resnet50_sparse.onnx", batch_size=32)
print(f"Total seconds ran: {results['benchmark_result']['seconds_ran']:.2f} s")
print(f"Iterations: {results['benchmark_result']['iterations']}")


2025-06-14 16:18:35 deepsparse.benchmark.helpers INFO     Thread pinning to cores enabled
2025-06-14 16:18:40 deepsparse.benchmark.benchmark_model INFO     deepsparse.engine.Engine:
	onnx_file_path: resnet50_sparse.onnx
	batch_size: 32
	num_cores: 20
	num_streams: 1
	scheduler: Scheduler.default
	fraction_of_supported_ops: 1.0
	cpu_avx_type: avx512
	cpu_vnni: True
2025-06-14 16:18:40 deepsparse.utils.onnx INFO     Generating input 'input', type = float32, shape = [32, 3, 224, 224]
2025-06-14 16:18:40 deepsparse.benchmark.benchmark_model INFO     Starting 'singlestream' performance measurements for 10 seconds


Total seconds ran: 10.06 s
Iterations: 124
